In [1]:
# Cell 1: Setup and imports
import fitz  # PyMuPDF - the PDF reader
import pandas as pd
import os
from pathlib import Path

# Set up paths relative to our project structure
# Since this notebook is inside the 'notebooks' folder,
# we go one level up (..) to reach the project root
PROJECT_ROOT = Path("..").resolve()
RAW_DATA = PROJECT_ROOT / "data" / "raw"

# List what we have
print("Project root:", PROJECT_ROOT)
print("\nFiles in data/raw:")
for f in RAW_DATA.iterdir():
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}  ({size_kb:.1f} KB)")

Project root: C:\Users\riyaw\OneDrive\Desktop\SpendScope

Files in data/raw:
  Redacted bank statement 1.pdf  (580.2 KB)
  redacted Bank statement 2.pdf  (675.3 KB)
  Redacted Bank statement 3.pdf  (303.6 KB)


In [2]:
# Cell 2: Open one PDF and inspect the raw text structure
# WHY: We need to see how Lloyds formats their PDF text layer.
# Every parser decision we make later depends on understanding this pattern.

pdf_files = sorted(RAW_DATA.glob("*.pdf"))
test_pdf = pdf_files[0]  # Pick the first file
print(f"Inspecting: {test_pdf.name}\n")

doc = fitz.open(str(test_pdf))
print(f"Number of pages: {len(doc)}\n")

# Extract text from page 1 and print every line with its index number
page = doc[0]
text = page.get_text()
lines = text.split('\n')

print(f"Total lines on page 1: {len(lines)}\n")
print("--- RAW TEXT (first 80 lines) ---")
for i, line in enumerate(lines[:80]):
    print(f"  L{i:03d}: {repr(line)}")

doc.close()

Inspecting: Redacted bank statement 1.pdf

Number of pages: 5

Total lines on page 1: 229

--- RAW TEXT (first 80 lines) ---
  L000: 'Page 1 of 5'
  L001: 'logo, Lloyds Bank.'
  L002: '10 March 2026'
  L003: 'If you think something is incorrect, please contact us on 0345 300 0000'
  L004: 'Lloyds Bank plc. Registered Office: 25 Gresham Street, London EC2V 7HN. Registered in England and Wales no. 2065 Lloyds Bank plc'
  L005: 'is authorised by the Prudential Regulation Authority and regulated by the Financial Conduct Authority and the Prudential Regulation'
  L006: 'Authority under registration number 119278.'
  L007: 'Document requested by:'
  L008: 'M'
  L009: 'Your Account'
  L010: 'Sort Code'
  L011: 'Account Number'
  L012: 'CLASSIC.'
  L013: '01 January 2026 to 31 January 2026.'
  L014: 'Money In.'
  L015: '£2,934.71.'
  L016: 'Balance on 01 January 2026.'
  L017: '£39.96.'
  L018: 'Money Out.'
  L019: '£2,315.56.'
  L020: 'Balance on 31 January 2026.'
  L021: '£659.11.'
  L022: '

In [3]:
# Cell 3: Extract and display ONE complete transaction block
# WHY: We need to see the exact repeating pattern for a single transaction
# so we know what to look for when building the parser.

doc = fitz.open(str(test_pdf))
page = doc[0]
text = page.get_text()
lines = text.split('\n')

# Find the first occurrence of the word "Date" that starts a transaction
# (not the column header, but the actual first transaction)
# We skip past the header area (first ~30 lines)
for i in range(30, len(lines)):
    if lines[i].strip() == 'Date' and i + 1 < len(lines):
        # Check if the next line looks like a date (e.g., "02 Jan 26.")
        next_line = lines[i + 1].strip()
        if 'Jan' in next_line or 'Feb' in next_line or 'Mar' in next_line:
            # Found it! Print the next 12 lines to see one full transaction
            print("=== ONE COMPLETE TRANSACTION BLOCK ===\n")
            for j in range(i, i + 12):
                label = ""
                if lines[j].strip() == 'Date':
                    label = "  <-- LABEL"
                elif lines[j].strip() == 'Description':
                    label = "  <-- LABEL"
                elif lines[j].strip() == 'Type':
                    label = "  <-- LABEL"
                elif 'Money In' in lines[j]:
                    label = "  <-- LABEL"
                elif 'Money Out' in lines[j]:
                    label = "  <-- LABEL"
                elif 'Balance' in lines[j]:
                    label = "  <-- LABEL"
                else:
                    label = "  <-- VALUE"
                print(f"  L{j:03d}: {repr(lines[j].strip())}{label}")
            break

doc.close()

print("\n\nThe pattern is: LABEL line, then VALUE line, repeating 6 times.")
print("Date -> value, Description -> value, Type -> value,")
print("Money In -> value, Money Out -> value, Balance -> value.")

=== ONE COMPLETE TRANSACTION BLOCK ===

  L035: 'Date'  <-- LABEL
  L036: '02 Jan 26.'  <-- VALUE
  L037: 'Description'  <-- LABEL
  L038: 'UBER UK RIDES.'  <-- VALUE
  L039: 'Type'  <-- LABEL
  L040: 'DEB.'  <-- VALUE
  L041: 'Money In (£)'  <-- LABEL
  L042: 'blank.'  <-- VALUE
  L043: 'Money Out (£)'  <-- LABEL
  L044: '8.30.'  <-- VALUE
  L045: 'Balance (£)'  <-- LABEL
  L046: '31.66.'  <-- VALUE


The pattern is: LABEL line, then VALUE line, repeating 6 times.
Date -> value, Description -> value, Type -> value,
Money In -> value, Money Out -> value, Balance -> value.


In [4]:
# Cell 4: Find and display an INCOME transaction (FPI type)
# WHY: We need to verify that income transactions follow the same 
# label-value pattern, and see how they differ from spending.

doc = fitz.open(str(test_pdf))

for page_num in range(len(doc)):
    page = doc[page_num]
    text = page.get_text()
    lines = text.split('\n')
    
    for i in range(len(lines)):
        if lines[i].strip() == 'Type' and i + 1 < len(lines):
            if lines[i + 1].strip().rstrip('.') == 'FPI':
                # Found an income transaction! Go back to find its Date label
                start = i - 4  # Date label should be 4 lines before Type
                print(f"=== INCOME TRANSACTION (page {page_num + 1}) ===\n")
                for j in range(start, start + 12):
                    print(f"  L{j:03d}: {repr(lines[j].strip())}")
                print()
                doc.close()
                raise StopIteration  # Break out of both loops

doc.close()

=== INCOME TRANSACTION (page 1) ===

  L047: 'Date'
  L048: '02 Jan 26.'
  L049: 'Description'
  L050: '.'
  L051: 'Type'
  L052: 'FPI.'
  L053: 'Money In (£)'
  L054: '10.00.'
  L055: 'Money Out (£)'
  L056: 'blank.'
  L057: 'Balance (£)'
  L058: '41.66.'



StopIteration: 

In [5]:
# Cell 5: Parse ALL transactions from ALL PDFs
# WHY: This is the core data extraction engine. It reads the label-value
# pattern we discovered and builds a list of structured transactions.

import re

def parse_lloyds_pdf(pdf_path):
    """
    Parse a Lloyds Bank PDF statement and extract all transactions.
    
    The Lloyds PDF text structure for each transaction is:
        'Date'          -> date value
        'Description'   -> description value
        'Type'          -> type code value
        'Money In (£)'  -> amount or 'blank.'
        'Money Out (£)' -> amount or 'blank.'
        'Balance (£)'   -> balance value
    """
    transactions = []
    doc = fitz.open(str(pdf_path))
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()
        lines = text.split('\n')
        
        i = 0
        while i < len(lines):
            line = lines[i].strip()
            
            # Look for the 'Date' label that starts a transaction
            if line == 'Date' and i + 1 < len(lines):
                date_val = lines[i + 1].strip().rstrip('.')
                
                # Verify it looks like a real date (e.g., "02 Jan 26")
                if not re.match(r'^\d{2}\s+(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+\d{2}$', date_val):
                    i += 1
                    continue
                
                # Now read ahead to find each label and its value
                j = i + 2  # Start after date value
                
                # Read Description
                desc_val = ''
                while j < len(lines) and lines[j].strip() != 'Description':
                    j += 1
                if j < len(lines):
                    j += 1  # Skip the 'Description' label
                    desc_val = lines[j].strip().rstrip('.')
                    if desc_val == 'blank' or desc_val == '':
                        desc_val = ''
                    j += 1
                
                # Read Type
                type_val = ''
                while j < len(lines) and lines[j].strip() != 'Type':
                    j += 1
                if j < len(lines):
                    j += 1  # Skip the 'Type' label
                    type_val = lines[j].strip().rstrip('.')
                    j += 1
                
                # Read Money In
                money_in = ''
                while j < len(lines) and not lines[j].strip().startswith('Money In'):
                    j += 1
                if j < len(lines):
                    j += 1  # Skip the label
                    val = lines[j].strip().rstrip('.')
                    if val != 'blank':
                        money_in = val.replace(',', '')
                    j += 1
                
                # Read Money Out
                money_out = ''
                while j < len(lines) and not lines[j].strip().startswith('Money Out'):
                    j += 1
                if j < len(lines):
                    j += 1  # Skip the label
                    val = lines[j].strip().rstrip('.')
                    if val != 'blank':
                        money_out = val.replace(',', '')
                    j += 1
                
                # Read Balance
                balance = ''
                while j < len(lines) and not lines[j].strip().startswith('Balance'):
                    j += 1
                if j < len(lines):
                    j += 1  # Skip the label
                    balance = lines[j].strip().rstrip('.').replace(',', '')
                    j += 1
                
                # Store the transaction
                transactions.append({
                    'date': date_val,
                    'description': desc_val if desc_val else '[REDACTED]',
                    'type': type_val,
                    'money_in': money_in,
                    'money_out': money_out,
                    'balance': balance,
                })
                
                i = j  # Jump ahead past this transaction
            else:
                i += 1
    
    doc.close()
    return transactions

# Parse all three PDFs
all_transactions = []
for pdf_path in sorted(RAW_DATA.glob("*.pdf")):
    txns = parse_lloyds_pdf(pdf_path)
    print(f"{pdf_path.name}: {len(txns)} transactions")
    all_transactions.extend(txns)

print(f"\nTotal transactions extracted: {len(all_transactions)}")
print(f"\nFirst 5:")
for t in all_transactions[:5]:
    print(f"  {t['date']} | {t['description'][:25]:<25} | {t['type']:<4} | In: {t['money_in']:>8} | Out: {t['money_out']:>8} | Bal: {t['balance']:>8}")

Redacted bank statement 1.pdf: 100 transactions
redacted Bank statement 2.pdf: 72 transactions
Redacted Bank statement 3.pdf: 21 transactions

Total transactions extracted: 193

First 5:
  02 Jan 26 | UBER UK RIDES             | DEB  | In:          | Out:     8.30 | Bal:    31.66
  02 Jan 26 | [REDACTED]                | FPI  | In:    10.00 | Out:          | Bal:    41.66
  02 Jan 26 | [REDACTED]                | FPI  | In:    50.00 | Out:          | Bal:    91.66
  02 Jan 26 | CHOPSTIX SOUTHAMPT        | DEB  | In:          | Out:     4.22 | Bal:    87.44
  02 Jan 26 | N I S STORE               | DEB  | In:          | Out:     3.29 | Bal:    84.15


In [6]:
# Cell 6: Validate extracted data against statement summaries
# WHY: If even one transaction is missing or duplicated, every 
# analysis downstream will be wrong. We verify against the 
# official totals printed on each statement.

# Official figures from your three statements:
expected = {
    'Jan': {'money_in': 2934.71, 'money_out': 2315.56, 'end_balance': 659.11},
    'Feb': {'money_in': 2303.86, 'money_out': 2231.97, 'end_balance': 731.00},
    'Mar': {'money_in': 50.00, 'money_out': 777.15, 'end_balance': 3.85},
}

for month, exp in expected.items():
    # Filter transactions for this month
    month_txns = [t for t in all_transactions if month in t['date']]
    
    # Calculate totals
    total_in = sum(float(t['money_in']) for t in month_txns if t['money_in'])
    total_out = sum(float(t['money_out']) for t in month_txns if t['money_out'])
    last_balance = float(month_txns[-1]['balance']) if month_txns else 0
    
    # Compare
    in_ok = abs(total_in - exp['money_in']) < 0.02
    out_ok = abs(total_out - exp['money_out']) < 0.02
    bal_ok = abs(last_balance - exp['end_balance']) < 0.02
    
    status = "ALL MATCH" if (in_ok and out_ok and bal_ok) else "MISMATCH"
    
    print(f"=== {month} === [{status}]")
    print(f"  Money In:    Extracted {total_in:>10,.2f}  Expected {exp['money_in']:>10,.2f}  {'OK' if in_ok else 'WRONG'}")
    print(f"  Money Out:   Extracted {total_out:>10,.2f}  Expected {exp['money_out']:>10,.2f}  {'OK' if out_ok else 'WRONG'}")
    print(f"  End Balance: Extracted {last_balance:>10,.2f}  Expected {exp['end_balance']:>10,.2f}  {'OK' if bal_ok else 'WRONG'}")
    print(f"  Transactions: {len(month_txns)}")
    print()

=== Jan === [ALL MATCH]
  Money In:    Extracted   2,934.71  Expected   2,934.71  OK
  Money Out:   Extracted   2,315.56  Expected   2,315.56  OK
  End Balance: Extracted     659.11  Expected     659.11  OK
  Transactions: 100

=== Feb === [ALL MATCH]
  Money In:    Extracted   2,303.86  Expected   2,303.86  OK
  Money Out:   Extracted   2,231.97  Expected   2,231.97  OK
  End Balance: Extracted     731.00  Expected     731.00  OK
  Transactions: 72

=== Mar === [ALL MATCH]
  Money In:    Extracted      50.00  Expected      50.00  OK
  Money Out:   Extracted     777.15  Expected     777.15  OK
  End Balance: Extracted       3.85  Expected       3.85  OK
  Transactions: 21



In [7]:
# Cell 7: Convert to pandas DataFrame and save as CSV
# WHY: A DataFrame gives us powerful tools for analysis, filtering,
# and transformation. The CSV becomes our single source of truth
# that the rest of the project builds on.

from datetime import datetime

# Convert to DataFrame
df = pd.DataFrame(all_transactions)

# Add a proper date column (ISO format: 2026-01-02)
# WHY: String dates are useless for sorting, filtering by month,
# or calculating time differences. We need real date objects.
df['date_iso'] = df['date'].apply(lambda d: datetime.strptime(d, '%d %b %y').strftime('%Y-%m-%d'))

# Add a direction column (IN or OUT)
df['direction'] = df.apply(
    lambda row: 'IN' if row['money_in'] else ('OUT' if row['money_out'] else 'UNKNOWN'), 
    axis=1
)

# Convert money columns to numeric (empty strings become NaN)
df['money_in'] = pd.to_numeric(df['money_in'], errors='coerce')
df['money_out'] = pd.to_numeric(df['money_out'], errors='coerce')
df['balance'] = pd.to_numeric(df['balance'], errors='coerce')

# Save to processed folder
output_path = PROJECT_ROOT / "data" / "processed" / "transactions_raw.csv"
df.to_csv(output_path, index=False)

# Display summary
print(f"Saved to: {output_path}")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['date_iso'].min()} to {df['date_iso'].max()}")
print(f"\nFirst 10 rows:")
df.head(10)

Saved to: C:\Users\riyaw\OneDrive\Desktop\SpendScope\data\processed\transactions_raw.csv
Shape: 193 rows x 8 columns
Columns: ['date', 'description', 'type', 'money_in', 'money_out', 'balance', 'date_iso', 'direction']
Date range: 2026-01-02 to 2026-03-09

First 10 rows:


,date,description,type,money_in,money_out,balance,date_iso,direction
0,02 Jan 26,UBER UK RIDES,DEB,NaN,8.30,31.66,2026-01-02,OUT
1,02 Jan 26,[REDACTED],FPI,10.0,NaN,41.66,2026-01-02,IN
2,02 Jan 26,[REDACTED],FPI,50.0,NaN,91.66,2026-01-02,IN
3,02 Jan 26,CHOPSTIX SOUTHAMPT,DEB,NaN,4.22,87.44,2026-01-02,OUT
4,02 Jan 26,N I S STORE,DEB,NaN,3.29,84.15,2026-01-02,OUT
5,02 Jan 26,GO SOUTH COAST,DEB,NaN,4.30,79.85,2026-01-02,OUT
6,02 Jan 26,UBER *TRIP,DEB,NaN,12.44,67.41,2026-01-02,OUT
7,02 Jan 26,UBR* PENDING.UBER,DEB,NaN,14.90,52.51,2026-01-02,OUT
8,05 Jan 26,UBR* PENDING.UBER,DEB,NaN,9.75,42.76,2026-01-05,OUT
9,05 Jan 26,UBR* PENDING.UBER,DEB,NaN,8.91,33.85,2026-01-05,OUT


In [8]:
# Cell 8: List all unique merchant descriptions
# WHY: Before writing normalisation rules, we need to see every
# description the bank gives us. This tells us how messy the data
# is and what patterns we need to handle.

desc_counts = df.groupby('description').agg(
    count=('description', 'size'),
    total_out=('money_out', 'sum'),
    total_in=('money_in', 'sum')
).sort_values('count', ascending=False)

print(f"Unique descriptions: {len(desc_counts)}\n")
print(f"{'Description':<35} {'Count':>5} {'Total Out':>10} {'Total In':>10}")
print("-" * 65)
for desc, row in desc_counts.iterrows():
    out = f"{row['total_out']:.2f}" if row['total_out'] > 0 else ""
    inc = f"{row['total_in']:.2f}" if row['total_in'] > 0 else ""
    print(f"{desc[:35]:<35} {int(row['count']):>5} {out:>10} {inc:>10}")

Unique descriptions: 59

Description                         Count  Total Out   Total In
-----------------------------------------------------------------
[REDACTED]                             34    1434.99    2160.91
GO SOUTH COAST                         22      54.00           
UBR* PENDING.UBER                      20     171.61           
NON-GBP TRANS FEE                       9      17.38           
NON-GBP PURCH FEE                       9       4.50           
Al Barakah Superst                      8      32.87           
WAITROSE STORES                         7      22.31           
ANTHEM HOMES LTD                        4    1440.00           
INTERNATIONAL FOOD                      4      39.66           
Google Canva AI Ph                      3       2.96           
CHA SHA                                 3      13.98           
The Body Shop                           3      33.64           
Google One                              3       5.16           
UBER UK RIDES

In [9]:
# Cell 9: Merchant Normalisation and Category Assignment
# WHY: Messy bank descriptions need to be mapped to clean merchant
# names and spending categories. Without this, every chart and 
# insight downstream will be fragmented and wrong.
#
# APPROACH: Two layers:
# 1. Regex rules to match messy descriptions -> clean merchant names
# 2. Category mapping to assign each merchant to a spending category

import re

# --- LAYER 1: NORMALISATION RULES ---
# Each tuple: (regex_pattern, clean_merchant_name)
# Order matters -- first match wins
# Patterns are case-insensitive

NORMALISATION_RULES = [
    # UBER ecosystem (6 different formats!)
    (r'UBR\*\s*PENDING\.?UBER', 'Uber'),
    (r'UBER\s*\*\s*TRIP', 'Uber'),
    (r'UBER\s+UK\s+RIDES', 'Uber'),
    (r'UBER\s*\*\s*TRIP\s+HELP', 'Uber'),
    (r'Uber\s+Payments\s+UK', 'Uber'),
    (r'UBER\s*\*\s*EATS', 'Uber Eats'),
    
    # Transport
    (r'GO\s+SOUTH\s+COAST', 'Go South Coast (Bus)'),
    (r'VOI\s+UK', 'Voi Scooters'),
    (r'Ant\*TrainPal', 'TrainPal'),
    (r'National\s+Express', 'National Express'),
    
    # Supermarkets & Groceries
    (r'SAINSBURYS', 'Sainsburys'),
    (r'WAITROSE', 'Waitrose'),
    (r'MARKS.SPENCER', 'Marks & Spencer'),
    (r'TESCO', 'Tesco'),
    (r'LIDL', 'Lidl'),
    (r'Al\s*Barakah', 'Al Barakah Superstore'),
    (r'INTERNATIONAL\s+FOOD', 'International Food Store'),
    (r'N\s+I\s+S\s+STORE', 'NIS Store'),
    (r'POUNDLAND', 'Poundland'),
    (r'SEOUL\s+PLAZA', 'Seoul Plaza'),
    
    # Eating Out
    (r'GREGGS', 'Greggs'),
    (r'SUBWAY', 'Subway'),
    (r'CHOPSTIX', 'Chopstix'),
    (r'TORTILLA', 'Tortilla'),
    (r'BOMBAY\s+SPICE', 'Bombay Spice'),
    (r'CHA\s+SHA', 'Cha Sha'),
    (r'Sravs\s+Kitchen', 'Sravs Kitchen'),
    (r'WING\s*STOP', 'Wingstop'),
    (r'Miss\s+Millies', 'Miss Millies'),
    (r'LS\s+Tap\s+and\s+Tandoor', 'LS Tap and Tandoor'),
    (r'MOURA', 'Mouras Restaurant'),
    (r'SQ\s*\*\s*KWACKERS', 'Kwackers'),
    (r'Oven', 'Oven'),
    (r'Scoops\s+Portswood', 'Scoops Portswood'),
    (r'SumUp\s*\*\s*Arch\s*Coffe', 'Arch Coffee'),
    (r'TST.*Nest\s*Coffee', 'Nest Coffee House'),
    (r'Southampton\s+Harbou', 'Southampton Harbour Hotel'),
    
    # Shopping
    (r'SHEIN', 'Shein'),
    (r'ZARA', 'Zara'),
    (r'PRIMARK', 'Primark'),
    (r'Space\s*NK', 'Space NK'),
    (r'The\s+Body\s+Shop', 'The Body Shop'),
    (r'MNK\*Photogenic', 'Photogenic'),
    
    # Subscriptions
    (r'Google\s+One', 'Google One'),
    (r'Google\s+Canva\s+AI', 'Canva (via Google)'),
    (r'APPLE\.COM/BILL', 'Apple Subscription'),
    (r'VOXI|WWW\.VOXI', 'Voxi Mobile'),
    
    # Housing
    (r'ANTHEM\s+HOMES', 'Anthem Homes (Rent)'),
    (r'INFORMATION\s+SERVIC', 'Information Services'),
    
    # Transfers
    (r'REMITLY', 'Remitly (Intl Transfer)'),
    (r'LNK\s+NOTEMACHINE', 'ATM Cash Withdrawal'),
    (r'FENGYUE\s+LIU', 'Personal Transfer'),
    (r'F\s+VUNDLA', 'Personal Transfer In'),
    
    # Professional
    (r'WACV', 'WACV Conference'),
    
    # Bank Fees
    (r'NON-GBP\s+TRANS\s+FEE', 'Foreign Transaction Fee'),
    (r'NON-GBP\s+PURCH\s+FEE', 'Foreign Purchase Fee'),
]

# --- LAYER 2: CATEGORY MAPPING ---
CATEGORY_MAP = {
    'Uber': 'Transport', 'Go South Coast (Bus)': 'Transport',
    'Voi Scooters': 'Transport', 'TrainPal': 'Transport',
    'National Express': 'Transport',
    
    'Sainsburys': 'Groceries', 'Waitrose': 'Groceries',
    'Marks & Spencer': 'Groceries', 'Tesco': 'Groceries',
    'Lidl': 'Groceries', 'Al Barakah Superstore': 'Groceries',
    'International Food Store': 'Groceries', 'NIS Store': 'Groceries',
    'Poundland': 'Groceries', 'Seoul Plaza': 'Groceries',
    
    'Greggs': 'Eating Out', 'Subway': 'Eating Out',
    'Chopstix': 'Eating Out', 'Tortilla': 'Eating Out',
    'Bombay Spice': 'Eating Out', 'Cha Sha': 'Eating Out',
    'Sravs Kitchen': 'Eating Out', 'Wingstop': 'Eating Out',
    'Miss Millies': 'Eating Out', 'LS Tap and Tandoor': 'Eating Out',
    'Mouras Restaurant': 'Eating Out', 'Kwackers': 'Eating Out',
    'Oven': 'Eating Out', 'Scoops Portswood': 'Eating Out',
    'Southampton Harbour Hotel': 'Eating Out',
    'Arch Coffee': 'Coffee & Cafe', 'Nest Coffee House': 'Coffee & Cafe',
    
    'Uber Eats': 'Food Delivery',
    
    'Shein': 'Shopping', 'Zara': 'Shopping', 'Primark': 'Shopping',
    'Space NK': 'Shopping', 'The Body Shop': 'Shopping',
    'Photogenic': 'Shopping',
    
    'Google One': 'Subscriptions', 'Canva (via Google)': 'Subscriptions',
    'Apple Subscription': 'Subscriptions', 'Voxi Mobile': 'Subscriptions',
    
    'Anthem Homes (Rent)': 'Rent', 'Information Services': 'Bills',
    
    'Remitly (Intl Transfer)': 'Transfers',
    'ATM Cash Withdrawal': 'Cash',
    'Personal Transfer': 'Transfers', 'Personal Transfer In': 'Transfers',
    
    'WACV Conference': 'Professional',
    
    'Foreign Transaction Fee': 'Bank Fees',
    'Foreign Purchase Fee': 'Bank Fees',
}


def normalise(description, direction):
    """
    Takes a raw bank description and returns (merchant_name, category).
    Direction is used to distinguish salary from restaurant spending
    (e.g., WINGSTOP as FPI = salary, WINGSTOP as DEB = eating out).
    """
    if description == '[REDACTED]':
        if direction == 'IN':
            return '[REDACTED]', 'Income'
        else:
            return '[REDACTED]', 'Transfers'
    
    for pattern, merchant in NORMALISATION_RULES:
        if re.search(pattern, description, re.IGNORECASE):
            category = CATEGORY_MAP.get(merchant, 'Other')
            
            # Special case: Wingstop as income = salary, not eating out
            if merchant == 'Wingstop' and direction == 'IN':
                return 'Wingstop (Salary)', 'Income'
            
            return merchant, category
    
    return description, 'Other'


# Apply to every row
df['merchant'] = df.apply(lambda row: normalise(row['description'], row['direction'])[0], axis=1)
df['category'] = df.apply(lambda row: normalise(row['description'], row['direction'])[1], axis=1)

# Check results
unmatched = df[df['category'] == 'Other']
print(f"Categorised: {len(df) - len(unmatched)} / {len(df)} ({(len(df) - len(unmatched)) / len(df) * 100:.1f}%)")
print(f"Uncategorised: {len(unmatched)}")
if len(unmatched) > 0:
    print("\nUncategorised descriptions:")
    for desc in unmatched['description'].unique():
        print(f"  - {desc}")

Categorised: 193 / 193 (100.0%)
Uncategorised: 0


In [10]:
# Cell 10: Verify Uber split and show spending by category
# WHY: We need to confirm Uber rides vs Uber Eats are separated,
# and see the overall spending picture before moving forward.

# Check Uber specifically
uber_txns = df[df['merchant'].str.contains('Uber|Wingstop', case=False)]
print("=== UBER & WINGSTOP VERIFICATION ===\n")
for _, row in uber_txns.iterrows():
    amt = f"Out: {row['money_out']:.2f}" if pd.notna(row['money_out']) else f" In: {row['money_in']:.2f}"
    print(f"  {row['date']} | {row['merchant']:<25} | {row['category']:<15} | {amt}")

# Spending by category (only outflows)
print("\n\n=== SPENDING BY CATEGORY ===\n")
spending = df[df['direction'] == 'OUT'].groupby('category').agg(
    total=('money_out', 'sum'),
    count=('money_out', 'size')
).sort_values('total', ascending=False)

total_spend = spending['total'].sum()

for cat, row in spending.iterrows():
    pct = row['total'] / total_spend * 100
    print(f"  {cat:<25} GBP {row['total']:>8,.2f}  ({int(row['count']):>2} txns)  {pct:>5.1f}%")

print(f"  {'─' * 60}")
print(f"  {'TOTAL':<25} GBP {total_spend:>8,.2f}  ({int(spending['count'].sum()):>2} txns)")

# Income summary
print("\n\n=== INCOME SUMMARY ===\n")
income = df[df['direction'] == 'IN'].groupby('category').agg(
    total=('money_in', 'sum')
).sort_values('total', ascending=False)

for cat, row in income.iterrows():
    print(f"  {cat:<25} GBP {row['total']:>8,.2f}")

=== UBER & WINGSTOP VERIFICATION ===

  02 Jan 26 | Uber                      | Transport       | Out: 8.30
  02 Jan 26 | Uber                      | Transport       | Out: 12.44
  02 Jan 26 | Uber                      | Transport       | Out: 14.90
  05 Jan 26 | Uber                      | Transport       | Out: 9.75
  05 Jan 26 | Uber                      | Transport       | Out: 8.91
  05 Jan 26 | Uber                      | Transport       | Out: 8.67
  12 Jan 26 | Uber                      | Transport       | Out: 6.90
  14 Jan 26 | Uber                      | Transport       | Out: 9.99
  19 Jan 26 | Uber                      | Transport       | Out: 7.70
  19 Jan 26 | Wingstop                  | Eating Out      | Out: 7.22
  19 Jan 26 | Uber                      | Transport       | Out: 7.93
  19 Jan 26 | Uber                      | Transport       | Out: 15.92
  19 Jan 26 | Uber                      | Transport       | Out: 9.52
  19 Jan 26 | Uber                      | Transpo

In [11]:
# Cell 11: Save the fully enriched dataset
# WHY: This is our single source of truth going forward.
# Every analysis, chart, and insight in the app will read from this file.

output_path = PROJECT_ROOT / "data" / "processed" / "transactions_enriched.csv"
df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"\nFinal dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print(f"\nSample:")
df[['date_iso', 'merchant', 'category', 'money_in', 'money_out', 'balance', 'direction']].head(10)

Saved to: C:\Users\riyaw\OneDrive\Desktop\SpendScope\data\processed\transactions_enriched.csv

Final dataset shape: 193 rows x 10 columns
Columns: ['date', 'description', 'type', 'money_in', 'money_out', 'balance', 'date_iso', 'direction', 'merchant', 'category']

Sample:


,date_iso,merchant,category,money_in,money_out,balance,direction
0,2026-01-02,Uber,Transport,NaN,8.30,31.66,OUT
1,2026-01-02,[REDACTED],Income,10.0,NaN,41.66,IN
2,2026-01-02,[REDACTED],Income,50.0,NaN,91.66,IN
3,2026-01-02,Chopstix,Eating Out,NaN,4.22,87.44,OUT
4,2026-01-02,NIS Store,Groceries,NaN,3.29,84.15,OUT
5,2026-01-02,Go South Coast (Bus),Transport,NaN,4.30,79.85,OUT
6,2026-01-02,Uber,Transport,NaN,12.44,67.41,OUT
7,2026-01-02,Uber,Transport,NaN,14.90,52.51,OUT
8,2026-01-05,Uber,Transport,NaN,9.75,42.76,OUT
9,2026-01-05,Uber,Transport,NaN,8.91,33.85,OUT


In [12]:
# Cell 12: Subscription Detector
# WHY: Forgotten subscriptions silently drain money. If we can
# automatically flag "you pay X every month for these services",
# that is immediately actionable insight for the user.
#
# APPROACH: Group by merchant, check if they appear in multiple
# months with similar amounts. If yes, flag as subscription.

# Add a month column for grouping
df['month'] = pd.to_datetime(df['date_iso']).dt.to_period('M')

# Only look at outgoing transactions with known merchants
spending = df[(df['direction'] == 'OUT') & (df['merchant'] != '[REDACTED]')].copy()

# For each merchant, find how many distinct months they appear in
# and what their typical transaction amount is
merchant_monthly = spending.groupby('merchant').agg(
    months_active=('month', 'nunique'),
    total_transactions=('money_out', 'size'),
    amounts=('money_out', list),
    avg_amount=('money_out', 'mean'),
    total_spent=('money_out', 'sum'),
).reset_index()

# A subscription is likely if:
# 1. It appears in 2+ months (our data only covers ~2.5 months)
# 2. The amounts are consistent (low variation)
# 3. Usually 1 transaction per month (not daily purchases)

def detect_subscription(row):
    if row['months_active'] < 2:
        return False
    
    amounts = row['amounts']
    avg = row['avg_amount']
    
    # Check if amounts are roughly consistent
    # (standard deviation is less than 20% of the mean)
    if avg == 0:
        return False
    
    import numpy as np
    std = np.std(amounts)
    variation = std / avg if avg > 0 else 999
    
    # Low transaction count relative to months = likely subscription
    # (e.g., 2-3 transactions over 2-3 months, not 20 transactions)
    txn_per_month = row['total_transactions'] / row['months_active']
    
    # Subscription criteria:
    # - Appears in 2+ months
    # - Relatively consistent amounts (variation < 20%)
    # - 1-2 transactions per month on average
    if variation < 0.20 and txn_per_month <= 2:
        return True
    
    return False

merchant_monthly['is_subscription'] = merchant_monthly.apply(detect_subscription, axis=1)

# Display results
subscriptions = merchant_monthly[merchant_monthly['is_subscription']].sort_values('total_spent', ascending=False)
non_subs = merchant_monthly[~merchant_monthly['is_subscription']].sort_values('total_spent', ascending=False)

print("=== DETECTED SUBSCRIPTIONS ===\n")
print(f"{'Merchant':<30} {'Avg/Month':>10} {'Months':>7} {'Total':>10}")
print("-" * 60)
monthly_sub_cost = 0
for _, row in subscriptions.iterrows():
    monthly_avg = row['total_spent'] / row['months_active']
    monthly_sub_cost += monthly_avg
    print(f"{row['merchant']:<30} GBP {monthly_avg:>7,.2f} {row['months_active']:>5}   GBP {row['total_spent']:>7,.2f}")

print(f"\n  Estimated monthly subscription cost: GBP {monthly_sub_cost:,.2f}")
print(f"  Estimated annual subscription cost:  GBP {monthly_sub_cost * 12:,.2f}")

print(f"\n\n=== NOT SUBSCRIPTIONS (top 10 by spend) ===\n")
for _, row in non_subs.head(10).iterrows():
    print(f"  {row['merchant']:<30} {int(row['total_transactions']):>3} txns  GBP {row['total_spent']:>8,.2f}")

=== DETECTED SUBSCRIPTIONS ===

Merchant                        Avg/Month  Months      Total
------------------------------------------------------------
Voxi Mobile                    GBP   12.00     2   GBP   24.00
Tortilla                       GBP    4.34     2   GBP    8.69
Apple Subscription             GBP    2.99     2   GBP    5.98
Google One                     GBP    1.72     3   GBP    5.16
Canva (via Google)             GBP    0.99     3   GBP    2.96

  Estimated monthly subscription cost: GBP 22.04
  Estimated annual subscription cost:  GBP 264.50


=== NOT SUBSCRIPTIONS (top 10 by spend) ===

  Anthem Homes (Rent)              4 txns  GBP 1,440.00
  Remitly (Intl Transfer)          1 txns  GBP   665.00
  WACV Conference                  1 txns  GBP   521.96
  Uber                            28 txns  GBP   274.59
  Information Services             1 txns  GBP   149.85
  ATM Cash Withdrawal              1 txns  GBP    80.00
  Sainsburys                       2 txns  GBP  

In [13]:
# Cell 13: Improved Subscription Detector
# WHY: Version 1 had false positives (Tortilla) and missed the
# biggest recurring cost (rent). We fix both by:
# 1. Adding a known-subscription list for obvious cases
# 2. Checking if individual amounts repeat, not just averages
# 3. Filtering out restaurants/food from auto-detection

# Known recurring charges (manually confirmed)
# WHY manual? Some subscriptions are obvious and trying to detect
# them algorithmically adds complexity without value.
KNOWN_SUBSCRIPTIONS = {
    'Anthem Homes (Rent)': 'Rent',
    'Voxi Mobile': 'Mobile Phone',
    'Google One': 'Cloud Storage',
    'Canva (via Google)': 'Design Tool',
    'Apple Subscription': 'Apple Services',
}

# Categories that should NEVER be flagged as subscriptions
NOT_SUBSCRIPTION_CATEGORIES = ['Eating Out', 'Groceries', 'Coffee & Cafe', 
                                'Transport', 'Shopping', 'Food Delivery']

def detect_subscription_v2(row):
    merchant = row['merchant']
    
    # Check known list first
    if merchant in KNOWN_SUBSCRIPTIONS:
        return True
    
    # Skip categories that are clearly not subscriptions
    # We need to look up the category from our main dataframe
    merchant_cats = df[df['merchant'] == merchant]['category'].unique()
    for cat in merchant_cats:
        if cat in NOT_SUBSCRIPTION_CATEGORIES:
            return False
    
    # Auto-detect: same rules as before but stricter
    if row['months_active'] < 2:
        return False
    
    amounts = row['amounts']
    avg = row['avg_amount']
    
    if avg == 0:
        return False
    
    import numpy as np
    std = np.std(amounts)
    variation = std / avg if avg > 0 else 999
    
    txn_per_month = row['total_transactions'] / row['months_active']
    
    if variation < 0.15 and txn_per_month <= 2:
        return True
    
    return False

merchant_monthly['is_subscription'] = merchant_monthly.apply(detect_subscription_v2, axis=1)

# Display improved results
subscriptions = merchant_monthly[merchant_monthly['is_subscription']].sort_values('total_spent', ascending=False)

print("=== DETECTED SUBSCRIPTIONS (v2) ===\n")
print(f"{'Merchant':<30} {'Type':<20} {'Avg/Month':>10} {'Total':>10}")
print("-" * 75)
monthly_sub_cost = 0
for _, row in subscriptions.iterrows():
    monthly_avg = row['total_spent'] / row['months_active']
    monthly_sub_cost += monthly_avg
    sub_type = KNOWN_SUBSCRIPTIONS.get(row['merchant'], 'Auto-detected')
    print(f"{row['merchant']:<30} {sub_type:<20} GBP {monthly_avg:>7,.2f} GBP {row['total_spent']:>7,.2f}")

print(f"\n  Monthly recurring costs:  GBP {monthly_sub_cost:,.2f}")
print(f"  Annual recurring costs:   GBP {monthly_sub_cost * 12:,.2f}")

=== DETECTED SUBSCRIPTIONS (v2) ===

Merchant                       Type                  Avg/Month      Total
---------------------------------------------------------------------------
Anthem Homes (Rent)            Rent                 GBP  720.00 GBP 1,440.00
Voxi Mobile                    Mobile Phone         GBP   12.00 GBP   24.00
Apple Subscription             Apple Services       GBP    2.99 GBP    5.98
Google One                     Cloud Storage        GBP    1.72 GBP    5.16
Canva (via Google)             Design Tool          GBP    0.99 GBP    2.96

  Monthly recurring costs:  GBP 737.70
  Annual recurring costs:   GBP 8,852.36


In [14]:
# Cell 14: Time-series aggregation (daily, weekly, monthly)
# WHY: The web app needs pre-computed summaries for charts.
# A user wants to see "how much did I spend each week" or
# "how does January compare to February" -- these aggregations
# make that possible without recalculating every time.

df['date_dt'] = pd.to_datetime(df['date_iso'])

# --- DAILY SPENDING ---
daily = df[df['direction'] == 'OUT'].groupby('date_iso').agg(
    total_out=('money_out', 'sum'),
    num_transactions=('money_out', 'size')
).reset_index()

print("=== DAILY SPENDING (top 10 heaviest days) ===\n")
top_days = daily.sort_values('total_out', ascending=False).head(10)
for _, row in top_days.iterrows():
    print(f"  {row['date_iso']}  GBP {row['total_out']:>8,.2f}  ({int(row['num_transactions'])} transactions)")

# --- WEEKLY SPENDING ---
df['week'] = df['date_dt'].dt.isocalendar().week.astype(int)
df['year_week'] = df['date_dt'].dt.strftime('%Y-W%V')

weekly = df[df['direction'] == 'OUT'].groupby('year_week').agg(
    total_out=('money_out', 'sum'),
    num_transactions=('money_out', 'size')
).reset_index()

print("\n\n=== WEEKLY SPENDING ===\n")
for _, row in weekly.iterrows():
    bar = '█' * int(row['total_out'] / 50)  # Simple visual bar
    print(f"  {row['year_week']}  GBP {row['total_out']:>8,.2f}  ({int(row['num_transactions']):>2} txns)  {bar}")

# --- MONTHLY SPENDING (excluding rent and transfers for lifestyle view) ---
lifestyle_cats = ['Transport', 'Groceries', 'Eating Out', 'Shopping', 
                  'Food Delivery', 'Coffee & Cafe', 'Subscriptions', 'Bank Fees']

lifestyle = df[(df['direction'] == 'OUT') & (df['category'].isin(lifestyle_cats))]

monthly_lifestyle = lifestyle.groupby(
    pd.to_datetime(lifestyle['date_iso']).dt.to_period('M')
).agg(
    total=('money_out', 'sum'),
    txns=('money_out', 'size')
).reset_index()
monthly_lifestyle.columns = ['month', 'total', 'txns']

print("\n\n=== MONTHLY LIFESTYLE SPENDING (excludes rent, transfers, one-offs) ===\n")
for _, row in monthly_lifestyle.iterrows():
    print(f"  {row['month']}  GBP {row['total']:>8,.2f}  ({int(row['txns'])} transactions)")

# --- DAY OF WEEK PATTERNS ---
df['day_of_week'] = df['date_dt'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

dow_spending = df[df['direction'] == 'OUT'].groupby('day_of_week').agg(
    total=('money_out', 'sum'),
    avg_per_day=('money_out', 'mean'),
    txns=('money_out', 'size')
)
dow_spending = dow_spending.reindex(day_order)

print("\n\n=== SPENDING BY DAY OF WEEK ===\n")
for day, row in dow_spending.iterrows():
    bar = '█' * int(row['total'] / 30)
    print(f"  {day:<12} GBP {row['total']:>8,.2f}  (avg GBP {row['avg_per_day']:>5,.2f} per txn)  {bar}")

=== DAILY SPENDING (top 10 heaviest days) ===

  2026-01-30  GBP   955.00  (4 transactions)
  2026-02-27  GBP   779.57  (5 transactions)
  2026-03-02  GBP   734.77  (10 transactions)
  2026-02-02  GBP   543.69  (18 transactions)
  2026-01-07  GBP   540.51  (4 transactions)
  2026-02-17  GBP   450.00  (1 transactions)
  2026-01-27  GBP   184.39  (4 transactions)
  2026-02-23  GBP   109.52  (8 transactions)
  2026-01-13  GBP   100.00  (1 transactions)
  2026-01-14  GBP    99.97  (7 transactions)


=== WEEKLY SPENDING ===

  2026-W01  GBP    47.45  ( 6 txns)  
  2026-W02  GBP   607.09  (20 txns)  ████████████
  2026-W03  GBP   305.17  (24 txns)  ██████
  2026-W04  GBP   150.29  (24 txns)  ███
  2026-W05  GBP 1,205.56  (15 txns)  ████████████████████████
  2026-W06  GBP   646.14  (28 txns)  ████████████
  2026-W07  GBP     8.61  ( 6 txns)  
  2026-W08  GBP   640.93  (14 txns)  ████████████
  2026-W09  GBP   936.29  (20 txns)  ██████████████████
  2026-W10  GBP   757.05  (14 txns)  ████████

ValueError: cannot convert float NaN to integer

In [15]:
# Cell 15: Fix day-of-week display (handle days with no spending)
print("=== SPENDING BY DAY OF WEEK (fixed) ===\n")
for day, row in dow_spending.iterrows():
    if pd.isna(row['total']):
        print(f"  {day:<12} No spending data")
    else:
        bar = '█' * int(row['total'] / 30)
        print(f"  {day:<12} GBP {row['total']:>8,.2f}  (avg GBP {row['avg_per_day']:>5,.2f} per txn)  {bar}")

=== SPENDING BY DAY OF WEEK (fixed) ===

  Monday       GBP 1,619.02  (avg GBP 21.03 per txn)  █████████████████████████████████████████████████████
  Tuesday      GBP   790.34  (avg GBP 52.69 per txn)  ██████████████████████████
  Wednesday    GBP   724.24  (avg GBP 24.97 per txn)  ████████████████████████
  Thursday     GBP   261.08  (avg GBP 11.35 per txn)  ████████
  Friday       GBP 1,930.00  (avg GBP 60.31 per txn)  ████████████████████████████████████████████████████████████████
  Saturday     No spending data
  Sunday       No spending data


In [16]:
# Cell 16: Save final enriched dataset with all new columns
# and print a complete Phase 1 summary

# Save the fully enriched dataset
output_path = PROJECT_ROOT / "data" / "processed" / "transactions_enriched.csv"
df.to_csv(output_path, index=False)

print("=" * 60)
print("  PHASE 1 COMPLETE: DATA ENGINE SUMMARY")
print("=" * 60)

print(f"\n  Dataset: {df.shape[0]} transactions, {df.shape[1]} columns")
print(f"  Period: {df['date_iso'].min()} to {df['date_iso'].max()}")
print(f"  Source: 3 Lloyds Bank PDF statements")

total_in = df['money_in'].sum()
total_out = df['money_out'].sum()
print(f"\n  Total Income:   GBP {total_in:>10,.2f}")
print(f"  Total Spending: GBP {total_out:>10,.2f}")
print(f"  Net:            GBP {total_in - total_out:>10,.2f}")

print(f"\n  Categories: {df['category'].nunique()}")
print(f"  Unique merchants: {df['merchant'].nunique()}")
print(f"  Subscriptions detected: {len(subscriptions)}")

sub_monthly = subscriptions['total_spent'].sum() / subscriptions['months_active'].max()
print(f"  Monthly subscription cost: GBP {sub_monthly:,.2f}")

print(f"\n  Files created:")
print(f"    data/processed/transactions_enriched.csv")
print(f"\n  Notebook: notebooks/01_data_exploration.ipynb")
print(f"\n{'=' * 60}")
print(f"  READY FOR PHASE 2: INTELLIGENCE LAYER")
print(f"{'=' * 60}")

  PHASE 1 COMPLETE: DATA ENGINE SUMMARY

  Dataset: 193 transactions, 15 columns
  Period: 2026-01-02 to 2026-03-09
  Source: 3 Lloyds Bank PDF statements

  Total Income:   GBP   5,288.57
  Total Spending: GBP   5,324.68
  Net:            GBP     -36.11

  Categories: 14
  Unique merchants: 54
  Subscriptions detected: 5
  Monthly subscription cost: GBP 492.70

  Files created:
    data/processed/transactions_enriched.csv

  Notebook: notebooks/01_data_exploration.ipynb

  READY FOR PHASE 2: INTELLIGENCE LAYER
